In [1]:
# Run once to install, then restart the Jupyter kernel.
%pip install "torch==2.7.1+cu128" "torchvision==0.22.1+cu128" "torchaudio==2.7.1+cu128" --index-url https://download.pytorch.org/whl/cu128
%pip install "pyannote.audio==3.3.2" "numpy==2.5.4" "huggingface-hub==0.34.4" "transformers==4.55.4" "datasets==4.0.0" python-dotenv ipywidgets
%pip check


Looking in indexes: https://download.pytorch.org/whl/cu128
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
No broken requirements found.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import torch
from dotenv import load_dotenv
from pyannote.audio import Pipeline
from pyannote.audio.core.task import Specifications, Problem, Resolution
from torch.torch_version import TorchVersion

load_dotenv()

allowed_classes = [TorchVersion, Specifications, Problem, Resolution]

with torch.serialization.safe_globals(allowed_classes):
    pipeline = Pipeline.from_pretrained(
        "pyannote/speaker-diarization-3.1",
        use_auth_token=os.getenv("HUGGING_FACE_TOKEN"),
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pipeline.to(device)
print(f"사용 장치: {device}")

사용 장치: cuda


In [5]:
import torchaudio
from pathlib import Path

audio_path = Path(
    r"C:\LUNA\whisper\audio\Forward_Base_Taking_a_Day_Off.mp3"
)

# MP3を一度だけ読み込む
waveform, sample_rate = torchaudio.load(str(audio_path))

# モノラル・16kHzに統一
waveform = waveform.mean(dim=0, keepdim=True)
if sample_rate != 16000:
    waveform = torchaudio.functional.resample(
        waveform, sample_rate, 16000
    )
    sample_rate = 16000

diarization = pipeline({
    "waveform": waveform,
    "sample_rate": sample_rate,
    "uri": audio_path.stem,
})

with audio_path.with_suffix(".rttm").open(
    "w", encoding="utf-8"
) as rttm:
    diarization.write_rttm(rttm)

In [6]:
import pandas as pd
rttm_path = Path(
    r"C:\LUNA\whisper\audio\Forward_Base_Taking_a_Day_Off.rttm"
)

df_rttm = pd.read_csv(
    rttm_path,
    sep=' ',
    header=None,
    names=['type', 'file', 'chnl', 'start', 'duration', 'C1', 'C2', 'speaker_id', 'C3', 'C4']
)

display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4
0,SPEAKER,Forward_Base_Taking_a_Day_Off,1,10.848,1.755,NaN,NaN,SPEAKER_00,NaN,NaN
1,SPEAKER,Forward_Base_Taking_a_Day_Off,1,12.822,9.754,NaN,NaN,SPEAKER_00,NaN,NaN
2,SPEAKER,Forward_Base_Taking_a_Day_Off,1,22.947,4.573,NaN,NaN,SPEAKER_00,NaN,NaN
3,SPEAKER,Forward_Base_Taking_a_Day_Off,1,27.605,2.363,NaN,NaN,SPEAKER_00,NaN,NaN
4,SPEAKER,Forward_Base_Taking_a_Day_Off,1,32.482,9.214,NaN,NaN,SPEAKER_00,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
78,SPEAKER,Forward_Base_Taking_a_Day_Off,1,357.747,0.658,NaN,NaN,SPEAKER_01,NaN,NaN
79,SPEAKER,Forward_Base_Taking_a_Day_Off,1,358.405,0.101,NaN,NaN,SPEAKER_02,NaN,NaN
80,SPEAKER,Forward_Base_Taking_a_Day_Off,1,358.507,2.312,NaN,NaN,SPEAKER_01,NaN,NaN
81,SPEAKER,Forward_Base_Taking_a_Day_Off,1,361.915,2.194,NaN,NaN,SPEAKER_01,NaN,NaN


In [7]:
df_rttm['end'] = df_rttm['start'] + df_rttm['duration']
display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end
0,SPEAKER,Forward_Base_Taking_a_Day_Off,1,10.848,1.755,NaN,NaN,SPEAKER_00,NaN,NaN,12.603
1,SPEAKER,Forward_Base_Taking_a_Day_Off,1,12.822,9.754,NaN,NaN,SPEAKER_00,NaN,NaN,22.576
2,SPEAKER,Forward_Base_Taking_a_Day_Off,1,22.947,4.573,NaN,NaN,SPEAKER_00,NaN,NaN,27.520
3,SPEAKER,Forward_Base_Taking_a_Day_Off,1,27.605,2.363,NaN,NaN,SPEAKER_00,NaN,NaN,29.968
4,SPEAKER,Forward_Base_Taking_a_Day_Off,1,32.482,9.214,NaN,NaN,SPEAKER_00,NaN,NaN,41.696
...,...,...,...,...,...,...,...,...,...,...,...
78,SPEAKER,Forward_Base_Taking_a_Day_Off,1,357.747,0.658,NaN,NaN,SPEAKER_01,NaN,NaN,358.405
79,SPEAKER,Forward_Base_Taking_a_Day_Off,1,358.405,0.101,NaN,NaN,SPEAKER_02,NaN,NaN,358.506
80,SPEAKER,Forward_Base_Taking_a_Day_Off,1,358.507,2.312,NaN,NaN,SPEAKER_01,NaN,NaN,360.819
81,SPEAKER,Forward_Base_Taking_a_Day_Off,1,361.915,2.194,NaN,NaN,SPEAKER_01,NaN,NaN,364.109


In [8]:
df_rttm["number"] = None
df_rttm.at[0, "number"] = 0

display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,Forward_Base_Taking_a_Day_Off,1,10.848,1.755,NaN,NaN,SPEAKER_00,NaN,NaN,12.603,0
1,SPEAKER,Forward_Base_Taking_a_Day_Off,1,12.822,9.754,NaN,NaN,SPEAKER_00,NaN,NaN,22.576,None
2,SPEAKER,Forward_Base_Taking_a_Day_Off,1,22.947,4.573,NaN,NaN,SPEAKER_00,NaN,NaN,27.520,None
3,SPEAKER,Forward_Base_Taking_a_Day_Off,1,27.605,2.363,NaN,NaN,SPEAKER_00,NaN,NaN,29.968,None
4,SPEAKER,Forward_Base_Taking_a_Day_Off,1,32.482,9.214,NaN,NaN,SPEAKER_00,NaN,NaN,41.696,None
...,...,...,...,...,...,...,...,...,...,...,...,...
78,SPEAKER,Forward_Base_Taking_a_Day_Off,1,357.747,0.658,NaN,NaN,SPEAKER_01,NaN,NaN,358.405,None
79,SPEAKER,Forward_Base_Taking_a_Day_Off,1,358.405,0.101,NaN,NaN,SPEAKER_02,NaN,NaN,358.506,None
80,SPEAKER,Forward_Base_Taking_a_Day_Off,1,358.507,2.312,NaN,NaN,SPEAKER_01,NaN,NaN,360.819,None
81,SPEAKER,Forward_Base_Taking_a_Day_Off,1,361.915,2.194,NaN,NaN,SPEAKER_01,NaN,NaN,364.109,None


In [10]:
for i in range(1, len(df_rttm)):
    if df_rttm.at[i, "speaker_id"] != df_rttm.at[i-1, "speaker_id"]:
        df_rttm.at[i, "number"] = df_rttm.at[i-1, "number"] + 1
    else:
        df_rttm.at[i, "number"] = df_rttm.at[i-1, "number"]

display(df_rttm.head(10))


,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,Forward_Base_Taking_a_Day_Off,1,10.848,1.755,NaN,NaN,SPEAKER_00,NaN,NaN,12.603,0
1,SPEAKER,Forward_Base_Taking_a_Day_Off,1,12.822,9.754,NaN,NaN,SPEAKER_00,NaN,NaN,22.576,0
2,SPEAKER,Forward_Base_Taking_a_Day_Off,1,22.947,4.573,NaN,NaN,SPEAKER_00,NaN,NaN,27.520,0
3,SPEAKER,Forward_Base_Taking_a_Day_Off,1,27.605,2.363,NaN,NaN,SPEAKER_00,NaN,NaN,29.968,0
4,SPEAKER,Forward_Base_Taking_a_Day_Off,1,32.482,9.214,NaN,NaN,SPEAKER_00,NaN,NaN,41.696,0
5,SPEAKER,Forward_Base_Taking_a_Day_Off,1,34.237,0.405,NaN,NaN,SPEAKER_01,NaN,NaN,34.642,1
6,SPEAKER,Forward_Base_Taking_a_Day_Off,1,36.160,0.759,NaN,NaN,SPEAKER_01,NaN,NaN,36.919,1
7,SPEAKER,Forward_Base_Taking_a_Day_Off,1,42.286,9.501,NaN,NaN,SPEAKER_00,NaN,NaN,51.787,2
8,SPEAKER,Forward_Base_Taking_a_Day_Off,1,52.225,4.944,NaN,NaN,SPEAKER_00,NaN,NaN,57.169,2
9,SPEAKER,Forward_Base_Taking_a_Day_Off,1,57.507,2.734,NaN,NaN,SPEAKER_00,NaN,NaN,60.241,2


In [11]:
df_rttm_grouped = df_rttm.groupby("number").agg(
    start=pd.NamedAgg(column='start', aggfunc='min'),
    end=pd.NamedAgg(column='start', aggfunc='max'),
    speaker_id=pd.NamedAgg(column='speaker_id', aggfunc='first')
)

display(df_rttm_grouped)

,start,end,speaker_id
number,,,
0,10.848,32.482,SPEAKER_00
1,34.237,36.160,SPEAKER_01
2,42.286,63.751,SPEAKER_00
3,80.609,122.645,SPEAKER_01
4,126.914,126.914,SPEAKER_00
5,128.585,259.366,SPEAKER_01
6,264.901,287.598,SPEAKER_02
7,289.538,289.538,SPEAKER_00
8,290.500,314.024,SPEAKER_02


In [12]:
df_rttm_grouped["duration"] = df_rttm_grouped["end"] - df_rttm_grouped["start"]
df_rttm_grouped = df_rttm_grouped.reset_index(drop=True)
display(df_rttm_grouped)

,start,end,speaker_id,duration
0,10.848,32.482,SPEAKER_00,21.634
1,34.237,36.160,SPEAKER_01,1.923
2,42.286,63.751,SPEAKER_00,21.465
3,80.609,122.645,SPEAKER_01,42.036
4,126.914,126.914,SPEAKER_00,0.000
5,128.585,259.366,SPEAKER_01,130.781
6,264.901,287.598,SPEAKER_02,22.697
7,289.538,289.538,SPEAKER_00,0.000
8,290.500,314.024,SPEAKER_02,23.524
9,321.938,327.305,SPEAKER_01,5.367


In [18]:
csv_path = Path(
    r"C:\LUNA\whisper\audio\Forward_Base_Taking_a_Day_Off_rttm.csv"
)

df_rttm_grouped.to_csv(
    csv_path,
    sep=',',
    index=False
)